In [2]:
# ============================================================================
# CELL 1: Setup with Hardcoded Paths (No More Path Confusion!)
# ============================================================================

import sys
import os

# Set working directory to project root
project_root = "/Users/swang/workspace/repos/qanda_clean1/qanda-v2"
os.chdir(project_root)
sys.path.append('.')

print(f"Working directory: {os.getcwd()}")

# Import after setting paths
from qanda_module.config import get_config
from qanda_module.database_clean import load_database_clean
import duckdb

# Override config paths manually
config = get_config("config.yaml")
print(f"Original DB path from config: {config.duck_db_name}")

# HARDCODE the correct paths
config.duck_db_name = "./data/qanda_mid_june9.duckdb"  # Fix the database name
config.input_json_path = "./data/input_json2"    # Fix the JSON path

print(f"✅ Fixed DB path: {config.duck_db_name}")
print(f"✅ Fixed JSON path: {config.input_json_path}")

# Check if database file exists with correct path
if os.path.exists(config.duck_db_name):
    file_size = os.path.getsize(config.duck_db_name)
    print(f"📁 Database found: {file_size:,} bytes ({file_size/1024/1024:.2f} MB)")
else:
    print("❌ Database still not found!")
    print("📁 Files in data directory:")
    for f in os.listdir("./data"):
        print(f"  - {f}")

Working directory: /Users/swang/workspace/repos/qanda_clean1/qanda-v2
Original DB path from config: ./data/qanda_mid_june9.duckdb
✅ Fixed DB path: ./data/qanda_mid_june9.duckdb
✅ Fixed JSON path: ./data/input_json2
📁 Database found: 36,712,448 bytes (35.01 MB)


In [3]:
from chromadb import PersistentClient

client = PersistentClient(path="./data/chroma_db_mid_june9")
collection = client.get_collection("qa_optimized")

# Simple test
results = collection.query(
    query_texts=["climate change"],
    n_results=3
)

print(f"✅ Retrieved {len(results['documents'][0])} results")
print(f"Sample speaker: {results['metadatas'][0][0].get('speaker_name')}")

✅ Retrieved 3 results
Sample speaker: JANE GOODALL


In [2]:
# ============================================================================
# CELL 2: Check if Current Database Has Enough Data for Debugging
# ============================================================================

con = duckdb.connect(config.duck_db_name)

print("📊 Checking database content...")

# Check tables and row counts
tables_info = []
try:
    tables = con.execute("SHOW TABLES").fetchall()
    
    for table_info in tables:
        table_name = table_info[0]
        try:
            count = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
            tables_info.append((table_name, count))
            print(f"✅ {table_name}: {count:,} rows")
        except Exception as e:
            print(f"❌ {table_name}: Error - {e}")
            
except Exception as e:
    print(f"❌ Cannot read tables: {e}")

# Check if we have the minimum needed for UI debugging
required_tables = ["fact_responses", "dim_panellist", "dim_episode", "dim_question"]
available_tables = [t[0] for t in tables_info]

missing_tables = [t for t in required_tables if t not in available_tables]

if missing_tables:
    print(f"\n❌ Missing required tables: {missing_tables}")
else:
    print(f"\n✅ All required tables present!")
    
    # Check if we have some panelist data for testing
    panelists_query = """
    SELECT speaker_name, COUNT(*) as response_count 
    FROM fact_responses 
    WHERE speaker_type = 3 AND speaker_name IS NOT NULL
    GROUP BY speaker_name 
    ORDER BY response_count DESC 
    LIMIT 10
    """
    
    try:
        top_panelists = con.execute(panelists_query).df()
        print(f"\n📈 Top 10 panelists by response count:")
        for _, row in top_panelists.iterrows():
            print(f"  - {row['speaker_name']}: {row['response_count']} responses")
            
        # Check if TANYA PLIBERSEK is in the data (from your original error)
        tanya_check = con.execute("""
            SELECT COUNT(*) as count 
            FROM fact_responses 
            WHERE speaker_name = 'TANYA PLIBERSEK' AND speaker_type = 3
        """).fetchone()[0]
        
        print(f"\n🔍 TANYA PLIBERSEK responses: {tanya_check}")
        
        if tanya_check > 0:
            print("✅ Great! We can debug with this data.")
        else:
            print("⚠️  TANYA PLIBERSEK not found, but we can use other panelists for debugging.")
            
    except Exception as e:
        print(f"❌ Error checking panelist data: {e}")

con.close()

# Decision point
if not missing_tables and any(count > 0 for _, count in tables_info):
    print(f"\n🎯 DECISION: Database is usable for debugging!")
    print(f"📝 We have {len(tables_info)} tables with data.")
    print(f"🚀 Let's proceed with debugging the UI issues.")
else:
    print(f"\n❌ Database needs rebuilding - too little data for debugging.")

IOException: IO Error: Could not set lock on file "/Users/swang/workspace/repos/qanda_clean1/qanda-v2/./data/qanda_may.duckdb": Conflicting lock is held in /Users/swang/.local/share/uv/python/cpython-3.12.10-macos-aarch64-none/bin/python3.12 (PID 30923) by user swang. See also https://duckdb.org/docs/stable/connect/concurrency

In [5]:
# ============================================================================
# CELL 3: Load System and Test Basic Functionality  
# ============================================================================

# Load the dataframes manually (bypassing the config issue)
dataframes = {}
con = duckdb.connect(config.duck_db_name)

dataframes["fact_responses"] = con.execute("SELECT * FROM fact_responses").df()
dataframes["dim_panellist"] = con.execute("SELECT * FROM dim_panellist").df()
dataframes["dim_episode"] = con.execute("SELECT * FROM dim_episode").df()
dataframes["dim_question"] = con.execute("SELECT * FROM dim_question").df()

print(f"✅ Loaded dataframes manually:")
for name, df in dataframes.items():
    print(f"  - {name}: {len(df)} rows")

# Create the helpers object manually
from qanda_module.legacy_imports import ImprovedQAHelpers
helpers = ImprovedQAHelpers(dataframes)

# Load the embedder
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(config.embedding_model_name)

print(f"✅ System components loaded successfully!")
print(f"📊 Helpers created with {len(helpers.df_reply)} responses")
print(f"🧠 Embedder loaded: {config.embedding_model_name}")

✅ Loaded dataframes manually:
  - fact_responses: 33942 rows
  - dim_panellist: 716 rows
  - dim_episode: 147 rows
  - dim_question: 1271 rows
Loaded 475 panelist profile URLs
Total panellist responses: 17815
Panellist responses with missing speaker_id: 0
✅ System components loaded successfully!
📊 Helpers created with 33942 responses
🧠 Embedder loaded: BAAI/bge-large-en-v1.5


In [6]:
# ============================================================================
# CELL 3.5: Verify All Paths and Configuration
# ============================================================================

print("🔍 CHECKING ALL CONFIGURATION PATHS")
print("=" * 60)

# 1. Database path (already confirmed but let's double-check)
print(f"📁 DuckDB Path: {config.duck_db_name}")
print(f"   Exists: {os.path.exists(config.duck_db_name)}")
if os.path.exists(config.duck_db_name):
    size = os.path.getsize(config.duck_db_name)
    print(f"   Size: {size:,} bytes ({size/1024/1024:.2f} MB)")

# 2. Chroma path and collection
print(f"\n🔍 ChromaDB Configuration:")
print(f"   Chroma Path: {config.chroma_path}")
print(f"   Collection Name: {config.collection_name}")
print(f"   Embedding Model: {config.embedding_model_name}")

# 3. Check if Chroma database exists
chroma_exists = os.path.exists(config.chroma_path)
print(f"   Chroma DB Exists: {chroma_exists}")

if chroma_exists:
    try:
        from chromadb import PersistentClient
        
        # Connect to ChromaDB
        client = PersistentClient(path=config.chroma_path)
        
        # List all collections
        collections = client.list_collections()
        collection_names = [c.name for c in collections]
        print(f"   Available Collections: {collection_names}")
        
        # Check if our specific collection exists
        if config.collection_name in collection_names:
            print(f"   ✅ Target collection '{config.collection_name}' found!")
            
            # Get collection info
            collection = client.get_collection(config.collection_name)
            count = collection.count()
            print(f"   Documents in collection: {count:,}")
            
            # Test a quick query to see if it works
            if count > 0:
                test_result = collection.get(limit=1, include=["documents", "metadatas"])
                print(f"   ✅ Collection is queryable")
                if test_result['metadatas']:
                    sample_metadata = test_result['metadatas'][0]
                    print(f"   Sample metadata keys: {list(sample_metadata.keys())}")
            else:
                print(f"   ⚠️  Collection is empty!")
                
        else:
            print(f"   ❌ Target collection '{config.collection_name}' NOT found!")
            print(f"   Available: {collection_names}")
            
    except Exception as e:
        print(f"   ❌ Error accessing ChromaDB: {e}")
else:
    print(f"   ❌ ChromaDB directory does not exist!")
    # Check what's actually in the directory
    chroma_parent = os.path.dirname(config.chroma_path)
    if os.path.exists(chroma_parent):
        print(f"   Contents of {chroma_parent}:")
        for item in os.listdir(chroma_parent):
            print(f"     - {item}")

# 4. Input JSON path (even though we're not using it now)
print(f"\n📁 JSON Input Path: {config.input_json_path}")
print(f"   Exists: {os.path.exists(config.input_json_path)}")

# 5. Model configuration
print(f"\n🤖 AI Configuration:")
print(f"   Chat Model: {config.chat_model_name}")
print(f"   Temperature: {config.temperature}")
print(f"   Retrieval K: {config.n_chunks_to_retrieve_k}")

# 6. OpenAI API key check
openai_key = os.getenv("OPENAI_API_KEY")
if openai_key:
    print(f"   OpenAI API Key: Set (length: {len(openai_key)} chars)")
else:
    print(f"   ❌ OpenAI API Key: NOT SET!")

print(f"\n📋 CONFIGURATION SUMMARY:")
config_ok = (
    os.path.exists(config.duck_db_name) and 
    os.path.exists(config.chroma_path) and
    openai_key is not None
)

if config_ok:
    print(f"✅ All essential paths and configs look good!")
    print(f"🚀 Ready to proceed with error debugging.")
else:
    print(f"❌ Some configuration issues found - fix before proceeding.")

🔍 CHECKING ALL CONFIGURATION PATHS
📁 DuckDB Path: ./data/qanda_may.duckdb
   Exists: True
   Size: 12,595,200 bytes (12.01 MB)

🔍 ChromaDB Configuration:
   Chroma Path: ./data/chroma_db_hf_may
   Collection Name: qa_chunks
   Embedding Model: BAAI/bge-large-en-v1.5
   Chroma DB Exists: True
   Available Collections: ['qa_chunks']
   ✅ Target collection 'qa_chunks' found!
   Documents in collection: 33,905
   ✅ Collection is queryable
   Sample metadata keys: ['speaker_name', 'question_flag', 'question_id', 'response_id', 'subtopic', 'episode_id', 'speaker_id', 'chunk_num', 'title', 'episode_date']

📁 JSON Input Path: ./data/input_json2
   Exists: True

🤖 AI Configuration:
   Chat Model: gpt-4o-mini
   Temperature: 0.3
   Retrieval K: 80
   OpenAI API Key: Set (length: 164 chars)

📋 CONFIGURATION SUMMARY:
✅ All essential paths and configs look good!
🚀 Ready to proceed with error debugging.


In [7]:
# ============================================================================
# CELL 4: Test the Exact Error Scenario
# ============================================================================

from qanda_module.data_processing import get_semantic_topic_matches, extract_panelist_name
from qanda_module.ui_gradio import get_panelist_episodes

print("🧪 TESTING EXACT ERROR SCENARIO")
print("=" * 60)

# Test with TANYA PLIBERSEK (from your original error)
test_panelist_display = "TANYA PLIBERSEK (8 eps)"  # Simulating dropdown value
print(f"🔍 Testing with: '{test_panelist_display}'")

try:
    # Step 1: Extract name (this should work)
    clean_panelist = extract_panelist_name(test_panelist_display)
    print(f"  ✅ Step 1 - Extracted name: '{clean_panelist}'")
    
    # Step 2: Test get_panelist_episodes (checking for date column error)
    print(f"  🔍 Step 2 - Testing get_panelist_episodes...")
    episodes_text = get_panelist_episodes(helpers, clean_panelist)
    print(f"  ✅ Step 2 - Episodes loaded: {len(episodes_text)} chars")
    
    # Step 3: Test semantic matching (checking for sklearn warnings)
    print(f"  🔍 Step 3 - Testing semantic matching...")
    import warnings
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        
        topics = get_semantic_topic_matches(helpers, clean_panelist, embedder, limit=16)
        
        if w:
            print(f"  ⚠️  Warnings caught: {len(w)}")
            for warning in w:
                print(f"    - {warning.category.__name__}: {warning.message}")
        else:
            print(f"  ✅ No warnings!")
            
    print(f"  ✅ Step 3 - Topics found: {len(topics)}")
    
    # Step 4: Test the date extraction from UI logic (this was failing)
    print(f"  🔍 Step 4 - Testing date extraction...")
    episodes_data = helpers.df_reply[
        (helpers.df_reply["speaker_name"] == clean_panelist) & 
        (helpers.df_reply["speaker_type"] == 3)
    ].merge(helpers.df_ep, left_on="episode_id", right_on="id")
    
    print(f"    Merge result shape: {episodes_data.shape}")
    print(f"    Columns: {list(episodes_data.columns)}")
    
    if not episodes_data.empty:
        # This is the line that was failing in the UI
        latest_date = episodes_data["date"].max()
        print(f"  ✅ Step 4 - Latest date: {latest_date}")
    else:
        print(f"  ⚠️  No episodes data after merge")
        
    print(f"\n🎉 ALL STEPS PASSED! No errors reproduced.")
    
except Exception as e:
    print(f"\n❌ ERROR REPRODUCED: {e}")
    import traceback
    traceback.print_exc()
    
    # Let's debug the specific failure
    print(f"\n🔍 DEBUGGING THE ERROR:")
    
    # Check what went wrong with the merge
    if 'episodes_data' in locals():
        print(f"episodes_data columns: {list(episodes_data.columns)}")
        print(f"episodes_data shape: {episodes_data.shape}")
        
    # Check individual dataframes
    print(f"helpers.df_reply columns: {list(helpers.df_reply.columns)}")
    print(f"helpers.df_ep columns: {list(helpers.df_ep.columns)}")

🧪 TESTING EXACT ERROR SCENARIO
🔍 Testing with: 'TANYA PLIBERSEK (8 eps)'
  ✅ Step 1 - Extracted name: 'TANYA PLIBERSEK'
  🔍 Step 2 - Testing get_panelist_episodes...
  ✅ Step 2 - Episodes loaded: 434 chars
  🔍 Step 3 - Testing semantic matching...
  ⚠️  Warnings caught: 3
    - RuntimeWarning: divide by zero encountered in matmul
    - RuntimeWarning: overflow encountered in matmul
    - RuntimeWarning: invalid value encountered in matmul
  ✅ Step 3 - Topics found: 8
  🔍 Step 4 - Testing date extraction...
    Merge result shape: (323, 21)
    Columns: ['date_x', 'title_x', 'host_x', 'subtopic', 'id_x', 'episode_id', 'episode_date', 'question_id', 'question_flag', 'question_num', 'speaker_type', 'speaker_id', 'speaker_name', 'content', 'id_y', 'date_y', 'title_y', 'host_y', 'url', 'description', 'ep_label']

❌ ERROR REPRODUCED: 'date'

🔍 DEBUGGING THE ERROR:
episodes_data columns: ['date_x', 'title_x', 'host_x', 'subtopic', 'id_x', 'episode_id', 'episode_date', 'question_id', 'question

Traceback (most recent call last):
  File "/Users/swang/workspace/repos/qanda_clean1/qanda-v2/.venv/lib/python3.12/site-packages/pandas/core/indexes/base.py", line 3812, in get_loc
    return self._engine.get_loc(casted_key)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "pandas/_libs/index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/hashtable_class_helper.pxi", line 7088, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas/_libs/hashtable_class_helper.pxi", line 7096, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'date'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/var/folders/n3/ghw6jjzn3cb62qbgbgm1qtf40000gn/T/ipykernel_24438/3423888004.py", line 54, in <module>
    latest_date = episodes_data["date"].max()
                  ~~~~~~~~~~~~~^^^^^^^^
  File "/Users

In [8]:
# Option 3: Be explicit about which columns we want
print(f"\nOption 3 - Explicit column selection:")

episodes_data_v3 = helpers.df_reply[
    (helpers.df_reply["speaker_name"] == clean_panelist) & 
    (helpers.df_reply["speaker_type"] == 3)
][['episode_id', 'speaker_name', 'speaker_type']].merge(
    helpers.df_ep[['id', 'date', 'title', 'url', 'ep_label']],  # Only episode columns we need
    left_on="episode_id", 
    right_on="id"
)

print(f"  Columns: {list(episodes_data_v3.columns)}")
print(f"  Can access: episodes_data_v3['date'].max()")

try:
    latest_date_v3 = episodes_data_v3["date"].max()
    print(f"  ✅ Success: {latest_date_v3}")
except Exception as e:
    print(f"  ❌ Failed: {e}")

print(f"\n🎯 Recommendation: Option 2 or 3 - clean 'date' column with no suffixes!")


Option 3 - Explicit column selection:
  Columns: ['episode_id', 'speaker_name', 'speaker_type', 'id', 'date', 'title', 'url', 'ep_label']
  Can access: episodes_data_v3['date'].max()
  ✅ Success: 2020-03-09

🎯 Recommendation: Option 2 or 3 - clean 'date' column with no suffixes!


In [9]:
# ============================================================================
# CELL 7: Test sklearn Warnings Fix
# ============================================================================

print("🔧 TESTING SKLEARN WARNINGS FIX")
print("=" * 40)

from qanda_module.data_processing import build_panelist_text_profile, ALL_SUBSTANTIVE_TOPICS
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Test with validation
clean_panelist = "TANYA PLIBERSEK"
profile = build_panelist_text_profile(helpers, clean_panelist)

print(f"Profile length: {len(profile)}")
print(f"Profile preview: {profile[:100]}...")

# Generate embeddings with detailed checking
profile_embedding = embedder.encode([profile])
topic_embeddings = embedder.encode(ALL_SUBSTANTIVE_TOPICS)

print(f"\nEmbedding diagnostics:")
print(f"  Profile shape: {profile_embedding.shape}")
print(f"  Profile range: {profile_embedding.min():.6f} to {profile_embedding.max():.6f}")
print(f"  Has NaN: {np.any(np.isnan(profile_embedding))}")
print(f"  Has Inf: {np.any(np.isinf(profile_embedding))}")
print(f"  All zeros: {np.all(profile_embedding == 0)}")

print(f"  Topic embeddings shape: {topic_embeddings.shape}")
print(f"  Topic range: {topic_embeddings.min():.6f} to {topic_embeddings.max():.6f}")

# Test cosine similarity with validation
import warnings

print(f"\nTesting cosine similarity:")

# OPTION A: Direct (should give warnings)
print(f"  Direct approach:")
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    similarities_direct = cosine_similarity(profile_embedding, topic_embeddings)[0]
    print(f"    Warnings: {len(w)}")

# OPTION B: With validation
print(f"  With validation:")
if (np.any(np.isnan(profile_embedding)) or np.any(np.isinf(profile_embedding)) or 
    np.any(np.isnan(topic_embeddings)) or np.any(np.isinf(topic_embeddings))):
    print(f"    ❌ Invalid embeddings detected - using fallback")
    similarities_safe = np.full(len(ALL_SUBSTANTIVE_TOPICS), 0.0)
else:
    # Check for problematic vectors (all zeros, etc.)
    profile_norm = np.linalg.norm(profile_embedding)
    if profile_norm == 0:
        print(f"    ❌ Zero-norm profile embedding - using fallback")
        similarities_safe = np.full(len(ALL_SUBSTANTIVE_TOPICS), 0.0)
    else:
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always")
            similarities_safe = cosine_similarity(profile_embedding, topic_embeddings)[0]
            print(f"    Warnings with validation: {len(w)}")

print(f"  Results comparison:")
print(f"    Direct: {len(similarities_direct)} values, {np.sum(~np.isnan(similarities_direct))} valid")
print(f"    Safe: {len(similarities_safe)} values, {np.sum(~np.isnan(similarities_safe))} valid")

🔧 TESTING SKLEARN WARNINGS FIX
Profile length: 1331
Profile preview: "DEPLORABLES" OR VOTERS JOURNALISTIC CONFIDENTIALITY NAURU SECRECY POLITICS & INFLUENCE OF MEDIA GOV...

Embedding diagnostics:
  Profile shape: (1, 1024)
  Profile range: -0.117121 to 0.250604
  Has NaN: False
  Has Inf: False
  All zeros: False
  Topic embeddings shape: (41, 1024)
  Topic range: -0.127980 to 0.271745

Testing cosine similarity:
  Direct approach:
    Warnings: 3
  With validation:
    Warnings with validation: 3
  Results comparison:
    Direct: 41 values, 41 valid
    Safe: 41 values, 41 valid


In [2]:
# ============================================================================
# CELL 1: Setup Environment
# ============================================================================

import sys
sys.path.append('..')

from qanda_module.config import setup_system
from qanda_module.data_processing import get_semantic_topic_matches, extract_panelist_name
from qanda_module.ui_gradio import get_panelist_episodes
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import warnings

In [4]:
print("🚀 Setting up system...")# Cell 2: Setup paths and initialize
import os
import sys

# Change to project root directory
os.chdir('..')  # Move from notebooks/ to project root
sys.path.append('.')

print(f"Working directory: {os.getcwd()}")
helpers, qa_chain, config = setup_system("config.yaml")
embedder = SentenceTransformer(config.embedding_model_name)
print("✅ System loaded successfully")
print(f"📊 Data loaded: {len(helpers.df_reply)} responses, {len(helpers.df_guests)} panelists")

Could not load table dim_panellist: Catalog Error: Table with name dim_panellist does not exist!
Did you mean "pg_namespace"?

LINE 1: SELECT * FROM dim_panellist
                      ^
Could not load table dim_question: Catalog Error: Table with name dim_question does not exist!
Did you mean "pg_description"?

LINE 1: SELECT * FROM dim_question
                      ^
Could not load table dim_episode: Catalog Error: Table with name dim_episode does not exist!
Did you mean "pg_namespace"?

LINE 1: SELECT * FROM dim_episode
                      ^


Could not load table fact_responses: Catalog Error: Table with name fact_responses does not exist!
Did you mean "pg_tablespace"?

LINE 1: SELECT * FROM fact_responses
                      ^


🚀 Setting up system...
Working directory: /Users/swang/workspace/repos/qanda_clean1


KeyError: 'fact_responses'

In [5]:
# ============================================================================
# DEBUG CELL: Check Available Dataframe Keys
# ============================================================================

import sys
import os
sys.path.append('.')

from qanda_module.database_clean import load_database_clean
from qanda_module.config import get_config

print(f"Working directory: {os.getcwd()}")

# Load config and database separately to see what's available
config = get_config("config.yaml")
print(f"Config loaded: {config.duck_db_name}")

# Load database and check keys
con, dataframes = load_database_clean(
    base_path=config.input_json_path,
    db_path=config.duck_db_name
)

print(f"✅ Database loaded successfully")
print(f"📊 Available dataframe keys: {list(dataframes.keys())}")
print(f"📊 Dataframe shapes:")
for key, df in dataframes.items():
    print(f"  - {key}: {df.shape}")

File not found: ./data/input_json2/panellists1.json
File not found: ./data/input_json2/questions1.json
File not found: ./data/input_json2/episodes1.json
File not found: ./data/input_json2/responses1.json


Working directory: /Users/swang/workspace/repos/qanda_clean1
Config loaded: ./data/qanda_may.duckdb
✅ Database loaded successfully
📊 Available dataframe keys: []
📊 Dataframe shapes:


In [6]:
# ============================================================================
# DEBUG: Check What Tables Are Actually in DuckDB
# ============================================================================

import duckdb
from qanda_module.config import get_config

config = get_config("config.yaml")
db_path = config.duck_db_name

print(f"Connecting to: {db_path}")
con = duckdb.connect(db_path)

# Check what tables actually exist
print("📊 Tables in database:")
tables_result = con.execute("SHOW TABLES").fetchall()
print(f"Raw result: {tables_result}")

if tables_result:
    for table_info in tables_result:
        table_name = table_info[0]
        try:
            count = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
            print(f"  ✅ {table_name}: {count:,} rows")
        except Exception as e:
            print(f"  ❌ {table_name}: Error - {e}")
else:
    print("❌ No tables found in database!")

con.close()

Connecting to: ./data/qanda_may.duckdb
📊 Tables in database:
Raw result: []
❌ No tables found in database!


SyntaxError: unterminated f-string literal (detected at line 74) (998837316.py, line 74)

In [8]:
# ============================================================================
# DEBUG: Check Database File Integrity
# ============================================================================

import os
import duckdb
from qanda_module.config import get_config

config = get_config("config.yaml")
db_path = config.duck_db_name

print(f"📁 Database file: {db_path}")
print(f"📍 Absolute path: {os.path.abspath(db_path)}")

if os.path.exists(db_path):
    # Check file size
    file_size = os.path.getsize(db_path)
    print(f"📏 File size: {file_size:,} bytes ({file_size/1024/1024:.2f} MB)")
    
    # Check if file is too small (likely corrupted)
    if file_size < 1024:  # Less than 1KB
        print("❌ File is suspiciously small - likely corrupted!")
    elif file_size < 1024 * 1024:  # Less than 1MB
        print("⚠️  File seems small for a database with transcript data")
    else:
        print("✅ File size looks reasonable")
    
    # Try to read file header to check if it's a valid DuckDB file
    try:
        with open(db_path, 'rb') as f:
            header = f.read(16)
            print(f"📄 File header (hex): {header.hex()}")
            print(f"📄 File header (ascii): {header}")
            
            # DuckDB files should start with specific magic bytes
            if header.startswith(b'DUCK'):
                print("✅ Valid DuckDB magic header found")
            else:
                print("❌ Invalid DuckDB header - file is corrupted!")
                
    except Exception as e:
        print(f"❌ Cannot read file header: {e}")
    
    # Try to connect and check if database is functional
    try:
        print("\n🔌 Testing database connection...")
        con = duckdb.connect(db_path)
        
        # Try a simple query
        result = con.execute("SELECT 1").fetchone()
        print(f"✅ Basic query works: {result}")
        
        # Check tables
        tables = con.execute("SHOW TABLES").fetchall()
        print(f"📊 Tables found: {len(tables)}")
        
        con.close()
        
    except Exception as e:
        print(f"❌ Database connection failed: {e}")
        print("🔧 Database is likely corrupted and needs to be rebuilt")
        
else:
    print("❌ Database file does not exist!")
    
    # Check what's in the data directory
    data_dir = os.path.dirname(db_path) or "."
    print(f"\n📁 Contents of {data_dir}:")
    try:
        for item in os.listdir(data_dir):
            item_path = os.path.join(data_dir, item)
            if os.path.isfile(item_path):
                size = os.path.getsize(item_path)
                print(f"  📄 {item}: {size:,} bytes")
            else:
                print(f"  📁 {item}/")
    except Exception as e:
        print(f"❌ Cannot list directory: {e}")

📁 Database file: ./data/qanda_may.duckdb
📍 Absolute path: /Users/swang/workspace/repos/qanda_clean1/data/qanda_may.duckdb
❌ Database file does not exist!

📁 Contents of ./data:
❌ Cannot list directory: [Errno 2] No such file or directory: './data'
